# Lab 5 — Monitoreo y data drift con Evidently

**Taller: MLOps en la práctica — del notebook a producción** · UNI
**Duración:** ~45 min · **Modalidad:** guiado + retos

## 🎯 Objetivos
1. Entender por qué **todo modelo en producción se degrada** (y por qué nadie te avisará).
2. Detectar **data drift** comparando datos de producción vs. datos de entrenamiento con Evidently.
3. Medir la degradación real del modelo y traducirla a **impacto en soles**.
4. Definir una política de alerta y re-entrenamiento (el "contrato de mantenimiento" del modelo).

## La historia (basada en hechos que te van a pasar)

Es marzo de 2026. Tu modelo de churn lleva meses en producción y el dashboard del call center sigue verde: la API responde en 40 ms, cero errores 500. Todos felices.

Pero en el trimestre pasaron cosas: AndesTel **subió tarifas 18%**, lanzó una **campaña agresiva de captación** (muchos clientes nuevos) y la red tuvo **más caídas**. El modelo fue entrenado en un mundo que ya no existe. ¿Sigue funcionando? *La infraestructura no te lo va a decir: el servicio responde perfecto... predicciones malas.*

In [ ]:
%pip install -q evidently==0.7.21  pyarrow

In [ ]:
# === Cargar: modelo champion + datos de referencia (2025) + datos "de producción" (2026) ===
# Necesitas: mlflow.db (Labs 2-3), churn_telco_peru.csv y churn_telco_peru_nuevos.csv
import os
import pandas as pd
import mlflow

URL_BASE = ""  # opcional: URL raw del repo del taller para descargar los CSV

for archivo in ["churn_telco_peru.csv", "churn_telco_peru_nuevos.csv"]:
    if not os.path.exists(archivo):
        if URL_BASE:
            pd.read_csv(f"{URL_BASE}/{archivo}").to_csv(archivo, index=False)
        else:
            raise FileNotFoundError(f"Falta {archivo}: súbelo o define URL_BASE")

mlflow.set_tracking_uri("sqlite:///mlflow.db")
modelo = mlflow.sklearn.load_model("models:/churn-andestel@champion")

FEATURES = ["edad", "meses_antiguedad", "cargo_mensual_soles", "gb_datos_mes",
            "minutos_llamadas_mes", "lineas_adicionales", "tickets_soporte_6m",
            "caidas_servicio_mes", "dias_ultimo_pago_vencido", "factura_electronica",
            "departamento", "plan", "tipo_contrato"]

ref = pd.read_csv("churn_telco_peru.csv").drop_duplicates(subset="id_cliente")   # mundo 2025 (entrenamiento)
prod = pd.read_csv("churn_telco_peru_nuevos.csv")                                # mundo 2026 (producción)
print(f"referencia: {len(ref)} filas | producción: {len(prod)} filas")

## 1. Primera señal: ¿cambió lo que predice el modelo?

Antes de mirar métricas (que requieren conocer el resultado real, y eso tarda meses en llegar), hay una señal disponible **desde el día uno**: la distribución de las predicciones.

In [ ]:
pred_ref = modelo.predict_proba(ref[FEATURES])[:, 1]
pred_prod = modelo.predict_proba(prod[FEATURES])[:, 1]

print(f"P(churn) promedio en 2025 (entrenamiento): {pred_ref.mean():.1%}")
print(f"P(churn) promedio en 2026 (producción):    {pred_prod.mean():.1%}")
print(f"Clientes 'riesgo alto' (>= 0.35): {(pred_ref >= 0.35).mean():.1%} -> {(pred_prod >= 0.35).mean():.1%}")

El modelo está marcando a **muchos más clientes** como riesgo alto. Dos hipótesis: (a) de verdad hay más fuga, (b) los datos cambiaron y el modelo alucina. Para distinguirlas necesitamos analizar el drift **feature por feature**. Hacerlo a mano son horas; **Evidently** lo automatiza.

## 2. Data drift con Evidently

Evidently compara dos datasets — **referencia** (con el que entrenaste) vs. **actual** (producción) — y aplica tests estadísticos por columna (Wasserstein, Jensen-Shannon, chi², según el tipo de variable). Genera un reporte HTML interactivo que puedes compartir con tu equipo.

In [ ]:
from evidently import Report
from evidently.presets import DataDriftPreset

reporte_drift = Report([DataDriftPreset()])
resultado = reporte_drift.run(
    reference_data=ref[FEATURES],
    current_data=prod[FEATURES],
)
resultado.save_html("reporte_drift.html")
print("✅ Generado: reporte_drift.html — descárgalo y ábrelo en el navegador")

# En notebook también puede verse inline (puede tardar en renderizar):
# resultado

In [ ]:
# === Extraer los resultados por código (para automatizar alertas) ===
import json

d = resultado.dict()
drifted = []
for m in d["metrics"]:
    cfg = m.get("config", {})
    if cfg.get("type", "").endswith("ValueDrift"):
        # score: distancia estadística (Wasserstein/Jensen-Shannon según tipo de columna)
        # drift si score > threshold del método
        drifted.append({
            "columna": cfg["column"],
            "metodo": cfg["method"],
            "score": round(float(m["value"]), 4),
            "umbral": cfg["threshold"],
            "drift": float(m["value"]) > cfg["threshold"],
        })

tabla_drift = pd.DataFrame(drifted).sort_values("score", ascending=False)
print(tabla_drift.to_string(index=False))

### 🗣️ Lectura del reporte (5 min)

Abre `reporte_drift.html` y responde:
1. ¿Qué columnas tienen mayor drift? ¿Coinciden con lo que sabemos del negocio (tarifas +18%, clientes nuevos, caídas de red)?
2. ¿Hay columnas SIN drift? ¿Qué te dice eso?
3. Si no supieras nada del negocio, ¿este reporte te habría contado la historia?

**💡 Para tu trabajo:** el reporte HTML es tu herramienta de comunicación con negocio: "no es que el modelo 'se malogró', es que ustedes subieron tarifas y el mundo cambió". Con evidencia, la conversación cambia.

## 3. La degradación real (cuando llega la verdad del terreno)

En este dataset de taller tenemos el lujo de conocer el churn real de 2026 (en la vida real llega con 1 trimestre de retraso — a esto se le llama *ground truth delay*). Midamos cuánto se degradó el modelo:

In [ ]:
from sklearn.metrics import roc_auc_score, recall_score, precision_score

UMBRAL = 0.35

def reporte_metricas(y, proba, etiqueta):
    pred = (proba >= UMBRAL).astype(int)
    print(f"{etiqueta:22s} AUC={roc_auc_score(y, proba):.4f}  "
          f"recall={recall_score(y, pred):.3f}  precision={precision_score(y, pred):.3f}")

reporte_metricas(ref["churn"], pred_ref, "2025 (entrenamiento)")
reporte_metricas(prod["churn"], pred_prod, "2026 (producción)")

In [ ]:
# === Traducción a soles (el idioma de tu gerencia) ===
pred_alto = (pred_prod >= UMBRAL).astype(int)
reales = prod["churn"].values

fn = int(((pred_alto == 0) & (reales == 1)).sum())   # churners que NO detectamos
fp = int(((pred_alto == 1) & (reales == 0)).sum())   # clientes fieles que llamamos por gusto

COSTO_CLIENTE_PERDIDO = 540   # S/ al año
COSTO_RETENCION = 15          # S/ por llamada+oferta

perdida_fn = fn * COSTO_CLIENTE_PERDIDO
gasto_fp = fp * COSTO_RETENCION
print(f"Falsos negativos: {fn} clientes -> pérdida ~S/ {perdida_fn:,}")
print(f"Falsos positivos: {fp} llamadas -> gasto  ~S/ {gasto_fp:,}")
print(f"En un lote de solo {len(prod)} clientes. AndesTel tiene 2.4 millones.")
print(f"Extrapolado: ~S/ {int((perdida_fn + gasto_fp) * 2_400_000 / len(prod)):,} de impacto")

## 4. De reporte a política: el contrato de mantenimiento del modelo

Un reporte que nadie mira no es monitoreo. Lo que convierte esto en MLOps es una **política explícita**, acordada y automatizable:

| Señal | Umbral de alerta | Acción |
|---|---|---|
| % de columnas con drift | > 30% | Revisar con negocio qué cambió |
| Drift en features top-5 de importancia | cualquiera | Investigación inmediata |
| Distribución de predicciones | media se mueve > 5 pts | Alerta al equipo DS |
| AUC con ground truth (mensual) | cae > 0.03 vs. baseline | Re-entrenar con datos recientes |
| Recall en producción | < 0.50 | Re-entrenar + revisar umbral |

El re-entrenamiento usa **el mismo pipeline del Lab 1-2**: por eso invertimos en reproducibilidad. Re-entrenar = correr el script con datos nuevos, comparar en MLflow contra el champion, y si gana, mover el alias. El círculo se cierra.

In [ ]:
# === Chequeo automatizable: la función que correría un job programado (cron / Airflow / GitHub Actions) ===
def chequeo_drift(ref_df, prod_df, features, umbral_share=0.30):
    resultado = Report([DataDriftPreset()]).run(ref_df[features], prod_df[features])
    d = resultado.dict()
    share = None
    for m in d["metrics"]:
        if m.get("config", {}).get("type", "").endswith("DriftedColumnsCount"):
            share = float(m["value"]["share"])
    alerta = share is not None and share > umbral_share
    return {"share_columnas_con_drift": round(share, 3) if share is not None else None,
            "alerta": bool(alerta)}

res = chequeo_drift(ref, prod, FEATURES)
print(json.dumps(res, indent=2))
if res["alerta"]:
    print("🚨 ALERTA: drift sobre el umbral -> notificar al equipo y evaluar re-entrenamiento")

## 🎯 Retos (15 min)

**Reto 1 (todos):** re-entrena el pipeline del Lab 2 usando SOLO datos combinados (2025 + 2026), evalúa sobre un split de 2026 y compara el AUC contra el champion actual. ¿Amerita mover el alias? Regístralo en MLflow como run `reentrenamiento-2026`.

**Reto 2 (todos):** genera el reporte de Evidently **solo con las 5 features de mayor importancia** del modelo. ¿Cambia tu diagnóstico?

**Reto 3 (avanzado):** simula monitoreo semanal: parte el dataset de producción en 4 lotes y ejecuta `chequeo_drift` por lote. Grafica la evolución del `share_columnas_con_drift`. ¿En qué "semana" habrías disparado la alerta?

**Reto 4 (avanzado):** usa `ClassificationPreset` de Evidently (`from evidently.presets import ClassificationPreset`) para generar el reporte de calidad del clasificador sobre 2026 y guárdalo como `reporte_clasificacion.html`.

## 📌 Lo que te llevas

- ✅ La infraestructura sana no implica modelo sano: hay que monitorear **los datos y las predicciones**, no solo la API.
- ✅ Evidently: comparar referencia vs. producción toma 5 líneas; el HTML es tu puente con negocio.
- ✅ El drift se traduce a **impacto en dinero** — así se consigue presupuesto para re-entrenar.
- ✅ Monitoreo = política explícita con umbrales y acciones, no un dashboard decorativo.
- ✅ El ciclo completo: reproducibilidad → tracking → registry → serving → monitoreo → **re-entrenar** → tracking...